# OFDM и Resource Grid

В основе физического уровня 5G NR лежит технология мультиплексирования с ортогональным частотным разделением каналов - CP-OFDM. 5G требует большой гибкости для обслуживания совершенно разных сценариев (от IoT с маленькими пакетами до потокового 4K-видео).

![Physical-layer-processing-for-5G-NR-PDSCH-and-PUSCH](images/Physical-layer-processing-for-5G-NR-PDSCH-and-PUSCH.png)

Для управления ресурсами используется **Resource Grid**. Всю доступную пропускную способность можно представить в виде двумерной матрицы, где:
*   Ось X - **время** - разделена на **Frames**, **Subframes**, **Slots** и **OFDM-символы**;
*   Ось Y - **частота** - разделена на **Subcarriers**.

Базовая станция на физическом уровне распределяет данные абонентов по ячейкам этой сетки, после чего передает их на обработку помехоустойчивым кодекам и далее через системную шину AXI в память. Именно размер и заполненность этой сетки определяют вычислительную нагрузку на аппаратную часть.

Структура Resource Grid состоит из блоков:
1.  **Resource Element** - самая маленькая и базовая единица сетки. Она равна 1 поднесущей по частоте на 1 OFDM-символ по времени. Один Resource Element может переносить комплексное число, представляющее модулированный символ (например, QPSK, 16-QAM, 256-QAM).
2.  **Resource Block** - в 5G ресурсный блок определяется только в частотной области и состоит из **12** последовательных поднесущих.
3.  **Нумерология - Numerology** - особенность 5G. В LTE расстояние между поднесущими всегда было 15 кГц. В 5G этот параметр гибкий: 15 кГц, 30 кГц, 60 кГц, 120 кГц и т.д. Чем шире поднесущая т.е. выше нумерология, тем короче OFDM-символ по времени. Это напрямую влияет на то, с какой скоростью кодеки базовой станции должны переваривать данные.

### [Sqimway](https://www.sqimway.com/lte_resource_grid.html)
![Resource Grid](images/sqimway.png)

В Resource Grid мы видим обозначения Resource Element

![Resource Element](images/res_elem.png)

1.  **PSCH / SSCH (Зеленый и светло-зеленый):** Сигналы синхронизации. С их помощью устройство понимает, что это сеть, и подстраивается под её ритм.
2.  **PBCH (Оранжевый):** Вещательный канал. Передает самую важную информацию о соте (MIB), без которой устройство не сможет подключиться.
3.  **RS (Красный):** **Reference Signals (Опорные сигналы).** Самые важные точки на сетке. По ним устройство оценивает качество связи. Если их много, связь стабильнее, но места для данных меньше.
4.  **PCFICH (Синий):** Канал, который сообщает устройству значение CFI (сколько символов занято управлением).
5.  **PHICH (Розовый):** Канал подтверждения приема (Hybrid-ARQ).
6.  **PDCCH (Желтый):** **Тот самый контрольный канал** В нем лежат команды сколько выделено ресурсов абоненту.
7.  **PDSCH (Белый/Прозрачный):** **Полезная нагрузка.** Сюда попадают пакеты по условно 100, 30, 10 байт.
8.  **Черный / Серый:** Неиспользуемые или зарезервированные ресурсы.

Параметры для настройки:

![sqimway parameters](images/sqimway_firstparam.png)

*   **Duplexing mode:** способ разделения приема и передачи.  Означает, что для загрузки (DL) и отдачи (UL) используются разные частоты.
*   **Channel bandwidth (1.4 MHz):** ширина канала. Высота сетки. 1.4 МГц — это самый узкий канал в LTE, он состоит из **6 ресурсных блоков (PRB)** или **72 поднесущих**. Чем шире канал, тем больше данных можно запихнуть.
*   **Cyclic prefix (Normal):** защитный интервал между символами, чтобы они не накладывались друг на друга из-за эха. В режиме Normal в одном слоте помещается **7 символов**.
*   **Tx antenna port (0 / 2):** количество антенн. От этого зависит, сколько будет красных квадратов (Reference Signals), так как каждой антенне нужны свои проверочные сигналы.
*   **Physical layer cell ID (0):** уникальный номер соты. От него зависит узор расположения сигналов синхронизации и проверочных сигналов, чтобы соседние вышки не светили в одни и те же ячейки.
*   **Control format indicator (CFI - 1):** он говорит, сколько первых символов в каждом подкадре отдано под управление (PDCCH). CFI=1 значит, что только первый символ занят командами, а остальные — данными.
*   **PHICH Ng factor / duration:** параметры канала, который подтверждает получение данных от устройствва (ACK/NACK). 

# py3gpp

Реализуем [калькулятор](calculator.py) для расчета нагрузки на железо на базе py3gg
(Для запуска калькулятора необходимо установить [py3gpp](https://github.com/catkira/py3gpp) )

(Полный код реализован в отдельном файле .py)
Разберем блоки кода:

In [ ]:
# Полная таблица MCS index table 2 for PDSCH (256QAM), TS 38.214 Table 5.1.3.1-2
# index: (modulation, R x 1024, Qm)
MCS_TABLE = {
    0:  ('qpsk',   120,   2), 1:  ('qpsk',   193,   2), 2:  ('qpsk',   308,   2),
    3:  ('qpsk',   449,   2), 4:  ('qpsk',   602,   2), 5:  ('qam16',  378,   4),
    6:  ('qam16',  434,   4), 7:  ('qam16',  490,   4), 8:  ('qam16',  553,   4),
    9:  ('qam16',  616,   4), 10: ('qam16',  658,   4), 11: ('qam64',  466,   6),
    12: ('qam64',  517,   6), 13: ('qam64',  567,   6), 14: ('qam64',  616,   6),
    15: ('qam64',  666,   6), 16: ('qam64',  719,   6), 17: ('qam64',  772,   6),
    18: ('qam64',  822,   6), 19: ('qam64',  873,   6), 20: ('qam256', 682.5, 8),
    21: ('qam256', 711,   8), 22: ('qam256', 754,   8), 23: ('qam256', 797,   8),
    24: ('qam256', 841,   8), 25: ('qam256', 885,   8), 26: ('qam256', 916.5, 8),
    27: ('qam256', 948,   8),
}

def _build_mcs_snr_thresholds(mcs_table, margin_db=3.0):
    thresholds = []
    for idx, (_, r_x1024, qm) in mcs_table.items():
        eff = qm * (r_x1024 / 1024.0)
        snr_req = 10 * math.log10(2 ** eff - 1) + margin_db
        thresholds.append((snr_req, idx))
    thresholds.sort()
    return thresholds

MCS_SNR_THRESHOLDS = _build_mcs_snr_thresholds(MCS_TABLE)

def snr_to_mcs(snr, thresholds=MCS_SNR_THRESHOLDS):
    mcs = thresholds[0][1]
    for req_snr, idx in thresholds:
        if snr >= req_snr:
            mcs = idx
        else:
            break
    return mcs

modulation, R_target, Qm = MCS_TABLE[mcs_index]
R = R_target / 1024.0  # скорость кодирования

Это [MCS Table (Modulation and Coding Scheme)](https://www.techplayon.com/5g-nr-modulation-and-coding-scheme-modulation-and-code-rate/) в нашем случае ограничились только таблицей 256QAM для uplink (PUSCH) и downlink (PDSCH). 

![MSC-TABLE](images/MCS-Table.png)

Благодаря этой таблице, меняя один индекс, программа автоматически пересчитывает спектральную эффективность ( Qm×R ). Это напрямую меняет количество выделяемых ресурсных блоков (PRB) и позволяет смоделировать, как изменится трафик на системной шине AXI при ухудшении или улучшении радиоприема.

Функция snr_to_mcs() вычисляет требуемый SNR для каждого индекса через спектральную эффективность:

eff = Qm × (R / 1024) → SNR_req = 10 · log10(2^eff − 1) + 3 дБ

Запас 3 дБ учитывает неидеальность реального приёмника. При дрейфе SNR функция возвращает наибольший MCS, порог которого не превышает текущий сигнал - переходы происходят между соседними индексами (например 14→15→16), а не скачками.

Таблица выдает программе 2 числа:

1) Порядок модуляции ($Q_m$)
*   Если MCS=0, то модуляция QPSK ($Q_m = 2$). Это значит, что в один RE мы можем запихнуть только 2 бита информации.
*   Если MCS=27, то 256QAM ($Q_m = 8$). Уже влезает 8 бит.
То есть при высоком MCS пакет "сожмется" и займет гораздо меньше ячеек (RE) и ресурсных блоков (PRB).

2) Скорость кодирования ($R$)

* В таблице записано число, которое нужно поделить на 1024. Например, для MCS=0 это $120 / 1024 = 0.117$. Это значит, что из всего объема переданных данных только 11.7% - это пакет, а остальные 88.3% - это защитные биты, которые кодек LDPC должен будет переварить, чтобы восстановить сигнал.


In [ ]:
## Таблица PRB по (BW МГц, SCS кГц), TS 38.101-1 Table 5.3.2-1 (FR1)
PRB_TABLE = {
    (5,15):25,  (10,15):52,  (15,15):79,  (20,15):106, (25,15):133,
    (40,15):216,(50,15):270, (10,30):24,  (15,30):38,  (20,30):51,
    (25,30):65, (30,30):78,  (40,30):106, (50,30):133, (60,30):162,
    (80,30):217,(100,30):273,(10,60):11,  (20,60):24,  (30,60):38,
    (40,60):51, (50,60):65,  (60,60):79,  (80,60):107, (100,60):135,
}

@dataclass
class CellConfig:
    duplex: str = "FDD"
    bw_mhz: int = 100
    scs_khz: int = 30
    tdd_pattern: str = "DDDSU"
    num_layers: int = 1
    tti_duration_ms: float = field(init=False)
    n_size_grid: int = field(init=False)
    mu: int = field(init=False)

    def __post_init__(self):
        self.tti_duration_ms = 1.0 / (self.scs_khz / 15)
        self.n_size_grid = PRB_TABLE[(self.bw_mhz, self.scs_khz)]
        self.mu = int(math.log2(self.scs_khz / 15))

## настройка сетки по конфигурации
carrier = py3gpp.nrCarrierConfig()
carrier.NSizeGrid = config.n_size_grid
carrier.SubcarrierSpacing = config.scs_khz

CellConfig - конфигурация соты. Все зависимые параметры вычисляются автоматически в __post_init__ по стандарту:

tti_duration_ms - длительность одного TTI (слота): 1 мс / (SCS / 15 кГц). Для 15 кГц → 1 мс, для 30 кГц → 0.5 мс, для 60 кГц → 0.25 мс.
n_size_grid - число PRB берётся из таблицы PRB_TABLE (TS 38.101-1 Table 5.3.2-1) по паре (BW, SCS). Для 100 МГц / 30 кГц → 273 PRB.
mu - нумерология: µ = log2(SCS / 15). Для 30 кГц → µ=1.

num_layers - число пространственных слоёв MIMO (1 = SISO, 2 = 2×2 MIMO). При num_layers=2 нагрузка на AXI удваивается, так как через шину проходит вдвое больше I/Q-данных за один TTI.

In [ ]:
## Планировщик
# Считаем емкость одного PRB: 12 поднесущих * 12 символов = 144 RE
bits_per_re = Qm * R
bits_per_prb = 144 * bits_per_re

# Считаем PRB с учетом +24 бит CRC
allocated_prb = max(1, math.ceil((packet_size_bytes * 8 + 24) / bits_per_prb))

# Выделяем ресурсы со сдвигом start_prb (для работы нескольких абонентов)
end_prb = min(start_prb + allocated_prb, carrier.NSizeGrid)
pdsch.PRBSet = list(range(start_prb, end_prb))

Определяет сколько Resource Elements дать пакету. 

In [ ]:
    ## определение кодека (LDPC BG1 BG2) (3GPP TS 38.212)
    # по стандарту 3GPP если Transport Block Size - размер данных - маленький то BG2
    # если большой то BG1
    A = packet_size_bytes * 8  # Считаем размер в битах

    if A <= 292:
        codec_type = "LDPC Base Graph 2"
    elif A <= 3824 and R <= 0.67:
        codec_type = "LDPC Base Graph 2"
    elif R <= 0.25:
        codec_type = "LDPC Base Graph 2"
    else:
        codec_type = "LDPC Base Graph 1"

Определяет какие будут нагружаться кодеки в зависимости от размера данных

In [ ]:
## Mapping 
result_indices = py3gpp.nrPDSCHIndices(carrier, pdsch)
# Извлечение массива из кортежа (особенность библиотеки)
ind = result_indices[0] if isinstance(result_indices, tuple) else result_indices 
num_re = len(ind)

Вычисление положение Resource Elements в сетке

In [ ]:
## разбиваем нагрузку на AXI по стадиям конвейера
## в реальном SoC каждый блок читает из Shared RAM и пишет результат обратно
num_re *= num_layers  # каждый доп. слой MIMO — независимый поток RE

tb_bytes      = math.ceil(A / 8)
cb_crc_bytes  = math.ceil((A + 48) / 8)  # A + CRC(24) + CB-CRC(24)
grid_bytes    = num_re * 4               # 16 бит I + 16 бит Q на RE

transactions = [
    AxiTransaction("Encoder LDPC", read_bytes=tb_bytes,    write_bytes=cb_crc_bytes),
    AxiTransaction("QAM mapper",   read_bytes=cb_crc_bytes, write_bytes=grid_bytes),
    AxiTransaction("Gridmapper",   read_bytes=grid_bytes,   write_bytes=grid_bytes),
]
axi_bus_bytes = sum(t.read_bytes + t.write_bytes for t in transactions)

В реальном SoC данные не текут напрямую от блока к блоку. Каждый этап конвейера обращается к Shared RAM дважды: читает входные данные и пишет результат обратно. AxiTransaction описывает одно такое обращение.

| Этап | Читает | Пишет |
|:--------:|:--------:|:--------:|
| Encoder LDPC | Transport Block | Закодированный блок|
| QAM mapper | Закодированные биты | Комплексные символы I/Q |
| Gridmapper | Символы I/Q | Финальная RE-сетка |

Суммарная нагрузка на AXI = сумма read_bytes + write_bytes по всем трём транзакциям.

#### MAC Scheduler
**Имитационная модель базовой станции**

Для проведения тестирования разработан генератор сценариев, который имитирует работу планировщика MAC-уровня. 
*   Симуляция проводится дискретными шагами по одному TTI. Длительность TTI определяется нумерологией конфигурации: для SCS 30 кГц — 0.5 мс.
*   Реализована **марковская модель радиоканала**: уровень сигнала (SNR) каждого абонента плавно изменяется во времени ($\pm 0.5$ дБ за TTI), что вызывает динамическое переключение кодеков и модуляции (MCS).
*   Реализован механизм **Fair Scheduling**: ограничивается максимальное количество ресурсов на одного пользователя (не более 40 PRB), чтобы в условиях лимита соты (273 PRB) обслуживать одновременно группу до 15 абонентов. Реализованы два режима дуплекса. В режиме FDD (Frequency Division Duplex) прием (UL) и передача (DL) работают одновременно на разных частотах - каждое направление имеет свои независимые 273 PRB. В режиме TDD (Time Division Duplex) DL и UL делят одну частоту по времени согласно паттерну слотов DDDSU - три слота только DL, один специальный слот (считается DL), один слот только UL. Функция get_slot_type(tti, pattern) возвращает тип каждого TTI, планировщик выделяет ресурсы только в активном для данного направления слоте.
*   Реализован механизм Power Control: каждому абоненту назначается штраф SNR пропорционально его псевдо-удалённости от вышки - distance_penalty = (ue_id % 5) × 1.5 дБ. Функция apply_power_control() вычитает штраф из измеренного SNR перед выбором MCS, что имитирует ситуацию, когда абонент на краю соты не может использовать высокие индексы MCS даже при формально приемлемом SNR канала.

#### PDCCH и Polar-кодер

Перед тем как передать данные абоненту по PDSCH, базовая станция отправляет ему команду управления по каналу PDCCH (Physical Downlink Control Channel) — сколько PRB выделено, какой MCS использовать. Эта команда кодируется Polar-кодером (не LDPC) — отдельным аппаратным блоком, который также нагружает AXI-шину.

In [ ]:
RE_PER_CCE = 72      # 1 CCE = 6 REG, 1 REG = 12 RE
PDCCH_DCI_BITS = 40  # типовой payload DCI format 1_0

def calculate_pdcch_resources(snr, dci_bits=PDCCH_DCI_BITS):
    if snr >= 20:   agg_level = 1
    elif snr >= 15: agg_level = 2
    elif snr >= 10: agg_level = 4
    elif snr >= 5:  agg_level = 8
    else:           agg_level = 16

    num_re    = agg_level * RE_PER_CCE
    A         = dci_bits + 24  # + CRC scrambled RNTI
    dci_bytes    = math.ceil(dci_bits / 8)
    coded_bytes  = math.ceil(A / 8)
    grid_bytes   = num_re * 4

    transactions = [
        AxiTransaction("Encoder Polar",    read_bytes=dci_bytes,  write_bytes=coded_bytes),
        AxiTransaction("PDCCH Gridmapper", read_bytes=coded_bytes, write_bytes=grid_bytes),
    ]
    return {
        "Aggregation Level": agg_level,
        "PDCCH REs": num_re,
        "AXI Bus Load": sum(t.read_bytes + t.write_bytes for t in transactions),
        "Transactions": transactions
    }

Aggregation Level определяет, сколько CCE (Control Channel Elements) базовая станция тратит на одну команду управления. Чем хуже SNR абонента — тем выше Aggregation Level, тем больше RE занимает PDCCH и тем выше нагрузка Polar-кодера на AXI-шину.

| SNR (дБ) | Aggregation Level | RE под PDCCH |
|----------|-------------------|--------------|
| ≥ 20     | 1                 | 72           |
| 15–20    | 2                 | 144          |
| 10–15    | 4                 | 288          |
| 5–10     | 8                 | 576          |
| < 5      | 16                | 1152         |

PDCCH_AXI_Bytes сохраняется в UeAllocation отдельно от AXI_Load_Bytes (PDSCH). Итоговая нагрузка на шину: Total_AXI = AXI_Load_Bytes + PDCCH_AXI_Bytes.

#### Аналитика метрик (Pandas)

**Анализ результатов и расчет Throughput**
Для обработки тысяч записей логов симуляции применяется **Pandas**. Формируется единый DataFrame, что позволяет рассчитать ключевые метрики сети за заданный период (например, 1 секунда / 2000 TTI):
1.  **Накопленный объем данных:** Суммирование переданных бит для каждого UE_ID.
2.  **Пользовательская скорость (User Throughput):** Деление накопленного объема на время симуляции. Мы наглядно видим, как высокий SNR обеспечивает скорость до 15-20 Мбит/с на абонента, а низкий SNR снижает ее до 1-2 Мбит/с.
3.  **Пропускная способность соты (Cell Capacity):** Суммарный объем данных всех абонентов за один слот.

#### График динамики абонентов

**Динамика подключений**

Выводится график, демонстрирующий плавное изменение количества активных пользователей в соте. Динамика задана вероятностной моделью: приток и отток абонентов происходит не мгновенно, а группами по 1-3 человека с низкой вероятностью на каждый TTI, что соответствует реальным условиям сети.

#### Вывод калькулятора

Калькулятор выводит конфигурацию

- **FDD** - Frequency Division Duplex - способ разделения приема/передачи. FDD позволяет одновременно и передавать и принимать данные (на разных частотах)
- **BW** - Bandwidth - полоса пропускания
- **SCS** - шаг поднесущих 
- **TTI** - интервал передачи

Затем калькулятор выводит массив с параметрами (пример для пакета 100 байт)

- **TTI** - слот времени (для 15 кГц = 1 мс, 30 кГц = 0.5 мс, 60 кГц = 0.25 мс, 120 кГц = 0.125 мс) в течение которого базовая станция передает данные пользователям на загрузку
- **Direction (DL/UL)** - направление передачи (Downlink - скачивание, Uplink - загрузка).
- **UE_ID** - уникальный идентификатор пользователя (User Equipment).
- **SNR** - отношение сигнал/шум для данного пользователя. Является показателем качества канала связи. Чем выше, тем лучше связь и тем более сложную модуляцию можно использовать. Низкое качество радиоканала деградирует производительность не только для абонента, но и для базовой станции, потому что вынуждает тратить больше ресурсов (аппаратных) на обслуживание малых объемов данных.
- **MCS** - Modulation and Coding Scheme - выбирается на основе [MCS Table (Modulation and Coding Scheme)](https://www.techplayon.com/5g-nr-modulation-and-coding-scheme-modulation-and-code-rate/). При идеальном сигнале (MSC 27) для передачи требуется минимальное кол-во PRB и минимальная загрузка на AXI. При обратной ситуации (MSC 0) из-за избыточного кодирования и низкой модуляции тот же объем данных требует примерно в 30 раз больше ресурсов сетки и соответственно большего трафика по шине AXI.
- **Ресурсы** - общее кол-во занятых Resource Elements (10 PRB * 12 Resource Elements. Пакет занимает 12 символов во времени<sup>1</sup>, т.е. 120*12=1440). Выглядит как геометрический формат выделенного участка сетки в виде `nPRB x nOFDMsymb x nLayers` (например, `4x12x1`).
- **PRBs** - количество выделенных ресурсных блоков. В одном PRB содержится 12 поднесущих.
- **Payload_Bits** - размер полезной нагрузки в битах.
- **Codec: LDPC Base Graph 1/2** - задаем порог для кодека, [ссылка на справки по выбору базового графа LDPC](https://sharetechnote.com/html/5G/5G_PDSCH.html) (п.2)<sup>2</sup>
- **AXI Load** - объем «сырых» радиоданных. Один Resource Element в памяти представлена как комплексное число: 16 бит для части I и 16 бит для части Q. Итого 4 байта на один Resource Element. Нужно чтобы понять выдержит ли AXI. Если шина AXI медленная, она не успеет прокачать эти 5760 байта за время одного слота (0.5 мс), и связь оборвется.

*Пример расчета:* Если пакет занял 10 PRB, он занимает $10 \times 12 \text{ поднесущих} \times 12 \text{ символов}^{1} = 1440 \text{ RE}$. Нагрузка составит $1440 \times 4 = \mathbf{5760 \text{ байт}}$. Если пропускная способность шины AXI недостаточна, она не успеет передать эти данные за время одного слота (0.5 мс), что приведет к переполнению буферов.

**Примечания**

<sup>1</sup> Почему из 14 возможных символов используем 12? (всего OFDM от 0 до 13) Отнимаем 1,2 или 3 символа на контроль (у нас 1 и 2 зарезервированы). Иногда 11, 12, 13 остаются пустыми для успешного переключения с передачи на прием, в нашем случае из-за FDD, который в отличие от TDD (Time Devision Duplex) не требует наличия защитного интервала в конце слота. Мы задаем pdsch.SymbolAllocation = [2, 12] - 2 - номер стартового символа, 12 - кол-во символов (2+12=14)


<sup>2</sup> Выбор базового графа определяется размером транспортного блока (A) и скоростью кодирования (R) в соответствии со следующими критериями. (Кратко говоря BS1 для больших пакетов, BS2 для малых пакетов)

$$
A \le 292 \rightarrow \text{LDPC base graph 2} \\
A \le 3824 \text{ and } R \le 0.67 \rightarrow \text{LDPC base graph 2} \\
R \le 0.25 \rightarrow \text{LDPC base graph 2} \\
\text{else} \rightarrow \text{LDPC base graph 1}
$$

Затем выводится:

**Пропускная способность соты**

Пример: 

* Средний объем переданных данных (DL) за один TTI: 2.4 КБ
* Пиковый объем переданных данных (DL) за один TTI:  5.47 КБ
* Реальная средняя скорость соты (DL Cell Throughput): 39.35 Мбит/с

**Статистика по Абонентам за время симуляции**

Показывает объем данных и скорость UP/DL для каждого UE_ID.

**Нагрузка на шину AXI**

Пример: 

* Средняя нагрузка: 57635.14 байт / TTI
* Пиковая (Максимальная) нагрузка: 275184 байт / TTI

PDSCH AXI - суммарный трафик от канала данных (три стадии LDPC→QAM→Grid) за всю симуляцию.
PDCCH AXI - суммарный трафик от канала управления (Polar-кодер). При плохом SNR абонентов доля PDCCH растёт - Aggregation Level увеличивается и Polar-кодер занимает больше места на шине.
Рекомендация архитектуры буфера - функция recommend_buffer_architecture() определяет этап конвейера с наибольшей долей AXI-трафика. Если один этап потребляет более 40% шины - рекомендуется выделить для него локальный SRAM-буфер вместо обращений к общей Shared RAM.

**Использование кодеков LDPC**

Пример: 

* LDPC Base Graph 1    44070
* LDPC Base Graph 2    27930

**Графики**
Resource Grid

![Resource](images/Resource Grid from calc.png)

График распределения PRB между абонентами в одном TTI. По оси X — номера PRB (0…N-1 в зависимости от BW/SCS), цвет ячейки — UE_ID абонента, которому выделен данный PRB. Серый (0) — свободные ресурсы. Позволяет наглядно увидеть, насколько плотно заполнена сетка и как Fair Scheduling распределяет ресурсы между абонентами.

Динамика абонентов в соте

![abonents](images/abonents.png)

(Вероятность изменения кол-ва абонентов 1%)


